In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle

from sklearn.model_selection import GridSearchCV, cross_val_score
#from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold

In [2]:
#前処理したデータ
data = pd.read_csv("preprocess.csv")

In [3]:
#Xとyに分割
X = data.drop(columns=["y"])
y = data["y"]

In [4]:
# KFoldクラスを初期化
kf = KFold(n_splits=7, shuffle=True, random_state=42)

In [5]:
# パラメータグリッドを定義
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

# GridSearchCVを初期化
grid_search = GridSearchCV(estimator=RandomForestRegressor(), param_grid=param_grid, 
                           cv=5, n_jobs=-1, scoring='neg_mean_absolute_error')

# グリッドサーチを実行
grid_search.fit(X, y)

# 最適なモデルとそのパラメータを取得
best_model = grid_search.best_estimator_
best_params = grid_search.best_params_

# 最適なパラメータを表示
print("Best Parameters:", best_params)

Best Parameters: {'max_depth': None, 'max_features': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 50}


In [6]:
#評価
cv_scores = cross_val_score(grid_search.best_estimator_, X, y, cv=5, scoring="neg_mean_absolute_error")

In [7]:
#評価の表示
print("Cross Validation Scores:", cv_scores)

Cross Validation Scores: [-3.29311475 -1.00025369 -1.82288733 -1.61015684 -0.40418505]


In [8]:
#評価平均値の表示
print("Mean Score:", cv_scores.mean())

Mean Score: -1.6261195324896753


In [9]:
# 最適なモデルを保存
with open("Apple_model.pickle", mode="wb") as fp:
    pickle.dump(best_model, fp)